# Task 1 — Inconsistencies and duplicates
Ghaydaa Medhat Mohamed Amer · 58-5281 · IET

In [1]:
import pandas as pd
raw = pd.read_csv('club_signups.csv', dtype=str)
df = raw.copy()
print(df.shape)
print(df.dtypes)

(39, 9)
signup_id       str
student_id      str
full_name       str
email           str
faculty         str
city            str
club            str
signed_up_at    str
fee_paid        str
dtype: object


## Part A.1 — Look first

In [2]:
for c in ['faculty','city','club','fee_paid']:
    print(df[c].value_counts(dropna=False), '\n')

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
MET                                       3
iet                                       3
Engineering and Materials Science         1
Pharma                                    1
met                                       1
Media Engineering and Technology          1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64 

city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO          1
El Giza        1
Name: count, dtype: int64 

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Chess Club     3
Football       3
Soccer         2
music          2
chess          2
debate club    1
Music         

## A.2 — Faculty, club, city
Mechanical normalisation first (strip, lower-case, collapse spaces), then an explicit dictionary.

In [3]:
def norm(s):
    return s.str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

for c in ['faculty','club','city']:
    df[c] = norm(df[c])

faculty_map = {
    'met':'MET', 'media engineering and technology':'MET',
    'iet':'IET', 'information engineering and technology':'IET',
    'ems':'EMS', 'engineering and materials science':'EMS',
    'pharmacy':'Pharmacy', 'pharma':'Pharmacy',
    'management':'Management',
}
club_map = {
    'robotics':'Robotics',
    'debate':'Debate', 'debating':'Debate', 'debate club':'Debate',
    'music':'Music',
    'football':'Football', 'soccer':'Football',
    'chess':'Chess', 'chess club':'Chess',
}
city_map = {
    'cairo':'Cairo',
    'giza':'Giza', 'el giza':'Giza',
    'alexandria':'Alexandria', 'alex':'Alexandria',
}
df['faculty'] = df['faculty'].map(faculty_map)
df['club'] = df['club'].map(club_map)
df['city'] = df['city'].map(city_map)

assert df[['faculty','club','city']].notna().all().all(), 'unmapped value'
assert set(df.faculty) <= {'MET','IET','EMS','Pharmacy','Management'}
assert set(df.club) <= {'Robotics','Debate','Music','Football','Chess'}
assert set(df.city) <= {'Cairo','Giza','Alexandria'}
for c in ['faculty','club','city']:
    print(df[c].value_counts(), '\n')

faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64 

club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64 

city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64 



## A.3 — Names and emails

In [4]:
df['full_name'] = (df['full_name'].str.strip().str.replace(r'\s+', ' ', regex=True).str.title())
df['email'] = df['email'].str.strip().str.lower()
df[['student_id','full_name','email']].drop_duplicates().sort_values('student_id')

,student_id,full_name,email
27,55-2639,Youssef Fathy,youssef.fathy@student.guc.edu.eg
19,55-2971,Salma Ibrahim,salma.ibrahim@student.guc.edu.eg
14,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg
5,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg
26,58-1089,Karim Kamel,karim.kamel@student.guc.edu.eg
12,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg
16,58-8701,Farida Ibrahim,farida.ibrahim@student.guc.edu.eg
17,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg
6,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg
1,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg


## A.4 — fee_paid to boolean

In [5]:
fee_map = {'yes':True,'y':True,'true':True,'1':True,'no':False,'n':False,'false':False,'0':False}
df['fee_paid'] = df['fee_paid'].str.strip().str.lower().map(fee_map)
assert df['fee_paid'].notna().all()
df['fee_paid'] = df['fee_paid'].astype(bool)
print(df['fee_paid'].dtype, df['fee_paid'].value_counts().to_dict())

bool {True: 27, False: 12}


## A.5 — signed_up_at
Two formats: ISO `YYYY-MM-DD HH:MM` and `DD/MM/YYYY HH:MM`. The slash format is day-first: values like `15/09/2026` and `16/09/2026` cannot be month-first (there is no month 15 or 16), and every date falls in September 2026.

In [6]:
s = raw['signed_up_at'].str.strip()
iso = s.str.match(r'^\d{4}-\d{2}-\d{2} ')
print('ISO rows:', iso.sum(), '| slash rows:', (~iso).sum())
print('max first number in slash rows:', s[~iso].str.split('/').str[0].astype(int).max())
parsed = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
parsed[iso] = pd.to_datetime(s[iso], format='%Y-%m-%d %H:%M')
parsed[~iso] = pd.to_datetime(s[~iso], format='%d/%m/%Y %H:%M')
df['signed_up_at'] = parsed
assert df['signed_up_at'].notna().all()
print(df['signed_up_at'].dtype, df['signed_up_at'].min(), df['signed_up_at'].max())

ISO rows: 25 | slash rows: 14
max first number in slash rows: 18
datetime64[ns] 2026-09-13 14:36:00 2026-09-18 23:36:00


## Part B.1 — Exact duplicates

In [7]:
n0 = len(df)
n_exact = df.duplicated().sum()
print('rows before:', n0, '| exact duplicates:', n_exact)
df = df.drop_duplicates().reset_index(drop=True)
print('rows after:', len(df))

rows before: 39 | exact duplicates: 3
rows after: 36


## B.2 — Same student, same club, submitted again
Keep the latest submission.

In [8]:
n1 = len(df)
df = (df.sort_values(['signed_up_at','signup_id'])
        .drop_duplicates(subset=['student_id','club'], keep='last')
        .sort_values('signup_id').reset_index(drop=True))
print('removed:', n1-len(df), '| rows left:', len(df))

removed: 4 | rows left: 32


## B.3 — Check

In [9]:
assert not df.duplicated(subset=['student_id','club']).any()
df.to_csv('club_signups_clean.csv', index=False)
print(df.shape); df

(32, 9)


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
2,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
3,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
4,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
5,S006,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-14 01:22:00,True
6,S007,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Robotics,2026-09-14 17:03:00,True
7,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32:00,True
8,S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics,2026-09-15 01:05:00,True
9,S010,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,Management,Alexandria,Chess,2026-09-15 03:12:00,False


## Extra — does the order of steps matter?
Removing duplicates *before* fixing the club spellings.

In [10]:
r = raw.copy()
a = r.duplicated().sum()
r2 = r.drop_duplicates()
b = r2.duplicated(subset=['student_id','club'], keep='last').sum()
print('on raw data -> exact duplicates found:', a, '| student+club duplicates found:', b)
print('after cleaning first -> exact duplicates found:', n_exact, '| student+club duplicates found:', n1-len(df))

on raw data -> exact duplicates found: 3 | student+club duplicates found: 2
after cleaning first -> exact duplicates found: 3 | student+club duplicates found: 4
